# 01 - Prepare and validate adapter data

Edit the adapter contract and examples below before enabling writes. This notebook creates separate train, validation, and test JSONL files, a JSON Schema, and `io.yaml`, then runs the repository validator.

The included rows are a format demonstration, not a production-quality dataset.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import yaml


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Open this notebook from the granite-switch-adapter-guide checkout")


ROOT = find_repo_root(Path.cwd().resolve())
ADAPTER_NAME = "my_adapter"
EXAMPLE_DIR = ROOT / "workspaces" / ADAPTER_NAME
WRITE_FILES = False
print("Repository:", ROOT)
print("Workspace:", EXAMPLE_DIR)

## Define the contract

Replace this example with one narrow function. Keep outputs deterministic and make every JSON output follow the same schema.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ["positive", "negative"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
    },
    "required": ["label", "confidence"],
    "additionalProperties": False,
}


def row(row_id: str, text: str, label: str) -> dict[str, str]:
    output = json.dumps({"label": label, "confidence": 1.0})
    return {"id": row_id, "input": text, "output": output}


train_rows = [
    row("train-001", "The service was excellent", "positive"),
    row("train-002", "The result was disappointing", "negative"),
]
validation_rows = [row("validation-001", "A very helpful response", "positive")]
test_rows = [row("test-001", "This did not solve my problem", "negative")]

assert {row["id"] for row in train_rows}.isdisjoint(row["id"] for row in validation_rows)
assert {row["id"] for row in train_rows}.isdisjoint(row["id"] for row in test_rows)

In [ ]:
schema_text = json.dumps(schema, indent=2).replace("\n", "\n  ")
io_yaml = f"""name: {ADAPTER_NAME}
model: ~
response_format: |
  {schema_text}
transformations: []
instruction: |
  Classify the input and return only the required JSON object.
parameters:
  max_completion_tokens: 64
  temperature: 0.0
sentence_boundaries: ~
"""
parsed_io = yaml.safe_load(io_yaml)
assert parsed_io["name"] == ADAPTER_NAME
assert json.loads(parsed_io["response_format"]) == schema
print(io_yaml)

## Write the files

Review the examples first. Change `WRITE_FILES` to `True` only when the contract is ready.

In [ ]:
def write_jsonl(path: Path, rows: list[dict[str, str]]) -> None:
    path.write_text("".join(json.dumps(row) + "\n" for row in rows), encoding="utf-8")


if WRITE_FILES:
    EXAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    write_jsonl(EXAMPLE_DIR / "train.jsonl", train_rows)
    write_jsonl(EXAMPLE_DIR / "validation.jsonl", validation_rows)
    write_jsonl(EXAMPLE_DIR / "test.jsonl", test_rows)
    (EXAMPLE_DIR / "schema.json").write_text(json.dumps(schema, indent=2) + "\n", encoding="utf-8")
    (EXAMPLE_DIR / "io.yaml").write_text(io_yaml, encoding="utf-8")
    print("Wrote:", EXAMPLE_DIR)
else:
    print("Dry run only. Set WRITE_FILES = True after editing the contract and examples.")

## Validate

The validation command runs only after files have been written.

In [ ]:
if WRITE_FILES:
    split_pairs = [
        ("train.jsonl", "validation.jsonl"),
        ("train.jsonl", "test.jsonl"),
        ("validation.jsonl", "test.jsonl"),
    ]
    for first, second in split_pairs:
        command = [
            sys.executable,
            "-m",
            "granite_adapter_guide",
            "validate-dataset",
            str(EXAMPLE_DIR / first),
            str(EXAMPLE_DIR / second),
        ]
        subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Validation skipped because WRITE_FILES is False")